In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing hemolytic peptide datasets (Bhatnagar et al.)


This notebook curates the hemotoxicity/hemolysis dataset from **Bhatnagar et al.** starting from the original Excel file. The main objective is to produce a clean, \
standardized table with `sequence` and a binary `label` suitable for downstream analysis and machine learning.

- **Toxic effect / endpoint:** hemolytic
- **Source:** Bhatnagar et al.
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset

The pipeline performs the following steps:
- **Loads the raw Excel dataset** from the `Hemotoxicity` sheet (skipping header/metadata rows).
- **Keeps only the relevant columns** (sequence + label) and renames them to a unified schema:
  - `sequence`
  - `label`
- **Maps label strings to binary values**:
  - `NonHemolytic` → 0
  - `Hemolytic` → 1
- **Checks duplicated sequences** and resolves them conservatively:
  - unique sequences are kept,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description file and appends QC statistics.
- **Exports outputs**:
  - `processed_hemolytic_dataset.csv`
  - `metadata.json`

In [2]:
name_source = "Bhatnagar et al."
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_Bhatnagar = pd.read_excel(f"{PATH_INPUT}/{name_source}/datasetBhatnagar.xlsx", sheet_name="Hemotoxicity", skiprows=1104)
df_Bhatnagar = df_Bhatnagar.iloc[:, :2]

In [4]:
df_Bhatnagar = df_Bhatnagar.rename(columns={
    "Sequence": "sequence",
    "Label": "label"
})

In [5]:
df_Bhatnagar["label"] = df_Bhatnagar["label"].replace({
    "NonHemolytic": 0,
    "Hemolytic": 1
})
df_Bhatnagar.shape

(756, 2)

- Checking duplicates

In [6]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_Bhatnagar, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [7]:
df_full.shape

(756, 2)

In [8]:
df_errors.shape

(0, 1)

- Working with metada

In [9]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [10]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_Bhatnagar)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2024,
 'last update date': datetime.datetime(2024, 10, 30, 0, 0),
 'download date': Timestamp('2025-06-25 00:00:00'),
 'file format': 'xlsx',
 'peptide property': 'antibacterial, antimicrobial, hemolytic, toxic',
 'dataset information': 'Concentration constants, Positive, Negative',
 'unit of measurement': 'µg/mL',
 'obtaining negative dataset': 'No information',
 'repository or server': 'Supplementary material of the paper',
 'publication': 'https://www.sciencedirect.com/science/article/pii/S1359511324002137?via%3Dihub',
 'number_of_raw_sequences': 756,
 'number_of_sequences_retained': 756,
 'number_of_positive_sequences': 167,
 'number_of_negative_sequences': 589,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [11]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [12]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)